## Evaluation dei sistemi IR
Bisogna trovare un modo per misurare la qualità dei sistemi IR, in modo da poterli confrontare tra loro e migliorare quelli esistenti. Tuttavia, valutare un sistema IR significa **capire se gli utenti sono soddisfatti dei risultati** che ottengono quando fanno una ricerca, e non è facile misurarlo direttamente. Esistono diversi indicatori indiretti in questo senso: 
- Se i risultati restituiti sono **rilevanti** per il bisogno informativo dell'utente
- (quindi) se l'utente **clicca molto** sui risultati restituiti (attenzione in questo caso però, potrebbero essere ingannati da titoli fuorvianti)
- se gli utenti comprano un prodotto dopo aver effettuato una ricerca (ad esempio su Amazon)
- se gli utenti in generale **tornano** a utilizzare il sistema IR in futuro, oppure se lo abbandonano subito

La soluzione più usata e ragionevole per valutare un sistema IR è, in generale, quella di **misurare quanto i risultati sono rilevanti**. Per misurare la rilevanza dei risultati, sono chiaramente necessari tre elementi fondamentali:
1. Una **collezione di documenti** (dataset) da cui estrarre i risultati
2. Un insieme di **query** che rappresentano i bisogni informativi degli utenti da testare
3. Un insieme di **giudizi di rilevanza** che indicano se i risultati restituiti per ogni query sono rilevanti o meno. Si noti che i giudizi di rilevanza sono spesso ottenuti da **lavoro umano** che quindi costano moltissimo (servono persone esperte per poter giudicare documenti di qualsiasi argomento), e non sono pienamente affidabili (un giudice potrebbe pensarla diversamente da un altro).

Questo insieme di elementi costituisce un **benchmark** per la valutazione del sistema IR.

Qui sotto una tabella con benchmark "storici" per la valutazione IR. NQrys è il numero di query, NDocs è il numero di documenti nella collezione, e Q-D RelAss è il numero di giudizi di rilevanza disponibili.

<img src="img/tab.png" width="500px">

Inizialmente, prima degli anni '90, dataset molto piccoli con pochi documenti e facili da gestire (si parla di MB). A partire da inizio '90 con l'avvento del web è nato **TREC** (Text REtrieval Conference), un benchmark più grande (dell'ordine di 2GB) nato per confrontare in modo standard i sistemi di IR. Oggi TREC si usa ancora, ma il numero di pagine web è cresciuto enormemente (si parla di centinaia di milioni di pagine) e quindi esistono benchmark ancora più grandi, dell'ordine di TB (come ClueWeb).

Fondamentale capire che **la rilevanza di un documento NON è rispetto alla query in sé, ma all'originale bisogno informativo dell'utente.** Es. Need == "devo pulire la piscina", query == "pool cleaner" -> un documento sarà rilevante solo se risolve il bisogno informativo, non semplicemente se contiene le parole "pool" e "cleaner". Se ad esempio un documento restituito ha pool e cleaner ma parla di un software per la pulizia dei dati, allora non è davvero rilevante per il bisogno informativo dell'utente, per questo servono giudizi di rilevanza umani per capirlo.

**Perché Valutare un sistema IR?** Per capire qual è il miglior sistema IR, e quindi rispondere a domande tipo:
- Qual è il miglior algoritmo?
- Qual è il miglior ranking (cosine similarity, etc..)
- Qual è il miglior preprocessing (stemming, stop word removal, etc..)
- Qual è il miglior weighting (TF-IDF, BM25, etc..)
- Dove è meglio tagliare la lista dei risultati restituiti (top 10, top 100, etc..)

### Costruire il benchmark: Gold Standard
Come detto, per valutare un sistema IR è necessario costruire un benchmark, ed in particolare quindi serve una **collezione etichettata** di documenti. Per crearla, si seguono i seguenti passi, data una collezione documentale:
1. **Genera delle query particolarmente rappresentative** dei bisogni tipici degli utenti
2. **Utilizza un sistema IR base per recuperare tanti documenti potenzialmente rilevanti** (che abbia quindi alta recall, anche a costo di bassa precision)
3. **Gli esperti umani decideranno, per ogni documento, se questo è rilevante o meno per la query** (giudizi di rilevanza). Le etichette così ottenute costituiscono il gold standard (o **ground truth**) per la valutazione del sistema IR. Si parla di gold standard perché costa un sacco fare sto passaggio.

Quindi creare il Gold Standard è costoso in quanto richiede moltissimo lavoro umano (si deve infatti valutare una collezione documentale rappresentativa e quindi molto grande), ma è necessario al fine di valutare in modo accurato un sistema IR. Esistono anche metodi più economici per creare un benchmark, ad esempio utilizzando i click degli utenti come giudizi di rilevanza, ne parliamo alla fine del md. 

NB. Tipicamente decisione binaria (rilevante o non rilevante), ma si potrebbero anche usare giudizi di rilevanza più sfumati (ad esempio da 0 a 3, dove 0 == non rilevante, 1 == poco rilevante, 2 == abbastanza rilevante, 3 == molto rilevante).

### Precision, Recall, F1-Score
Le metriche per eccellenza per valutare un sistema IR sono **precision** e **recall**. D'ora in poi consideriamo come rilevante un documento se è stato giudicato rilevante dagli esperti umani (gold standard).
- **Precision**: **Tra tutti i documenti restituiti dal sistema IR, quanti sono effettivamente rilevanti**? $$Precision = \frac{TP}{TP + FP}$$ dove TP == true positives (documenti restituiti e rilevanti), FP == false positives (documenti restituiti ma non rilevanti).
- **Recall**: **Tra tutti i documenti rilevanti, quanti sono stati effettivamente restituiti dal sistema IR?** $$Recall = \frac{TP}{TP + FN}$$ dove FN == false negatives (documenti non restituiti ma rilevanti).

Un modo per ricordare velocemente è che né precision né recall guardano ai true negatives (TN), ovvero ai documenti non restituiti e non rilevanti, e che Precision guarda solo ai "positivi" P alla fine.

<img src="img/prec_recall.png" width="400px">

Intuitivamente, avere alta Recall significa trovare quasi tutti i documenti rilevanti, mentre avere alta Precision significa che quasi tutti i documenti restituiti sono rilevanti. In generale, **c'è un trade-off tra precision e recall**: se restituisco pochi documenti, è più probabile che siano rilevanti (alta precision), ma rischio di non restituire molti documenti rilevanti (bassa recall). Se invece restituisco molti documenti, è più probabile che trovi tutti quelli rilevanti (alta recall), ma rischio di restituire anche molti non rilevanti (bassa precision).

Dipende anche dall'applicazione: es. se faccio un test che dice se una persona ha una malattia, è più importante avere alta recall (non voglio che alcun paziente con la malattia sia testato negativo), anche a costo di avere bassa precision (a costo di avere falsi positivi e quindi persone non malate che risultano positive al test). Se invece faccio un sistema di raccomandazione per un e-commerce, è più importante avere alta precision (voglio consigliare solo prodotti rilevanti), anche a costo di avere bassa recall (potrei non consigliare tutti i prodotti rilevanti, ma almeno quelli che consiglio sono rilevanti).

E l'**Accuracy**? $$Accuracy = \frac{TP + TN}{TP + FP + FN + TN}$$ intuitivamente rappresenta la percentuale di documenti correttamente classificati (sia rilevanti che non rilevanti) rispetto al totale dei documenti. **Tuttavia in contesti come IR non è molto utile in quanto la collezione è enorme e quindi i documenti non rilevanti (TN) sono tantissimi e domina il calcolo**. Quindi anche se il sistema IR restituisse pochissimi documenti rilevanti (TP), il numero di TN sarebbe così alto da far sembrare il sistema IR molto accurato.

Allo stesso modo l'**Error Rate** (percentuale di documenti classificati in modo errato) non è utile in quanto il TN al denominatore porta sempre ad avere un error rate molto basso, anche se il sistema IR è pessimo.
$$Error Rate = \frac{FP + FN}{TP + FP + FN + TN}$$

Torniamo al trade-off tra precision e recall:
- **avere alta precision e bassa recall** significa restituire pochi documenti, ma quasi tutti rilevanti -> **perdo documenti utili**
- **avere alta recall e bassa precision** significa restituire molti documenti, ma molti non rilevanti -> **perdo tempo a leggere documenti inutili**

<img src="img/prec_recall_to.png" width="400px">

L'ideale sarebbe avere sia Precision che Recall ad 1, ma è praticamente impossibile e quindi nella pratica si cerca di trovare un compromesso tra i due. Per questo motivo, spesso si usa una metrica che combina precision e recall, come ad esempio l'**F1-Score**, che è la media armonica tra precision e recall: $$\text{F1-Score} = \frac{2}{\frac{1}{P} + \frac{1}{R}} = \frac{2 P R}{P + R}$$

L'uso della media armonica ($H = \frac{n}{\frac{1}{x_1} + \dots + \frac{1}{x_n}}$) invece della media aritmetica è giustificato dal fatto che l'armonica penalizza maggiormente valori sbilanciati: se si ha altissima precision ma bassa recall (o viceversa), si vuole comunque avere un F1-Score basso (mentre se avessi usato la media aritmetica avrei avuto un valore più alto). Dal grafico seguente si vede come l'armonica sia la metrica più severa rispetto alle altre, ad eccezione del minimo.

<img src="img/f1.png" width="400px">

L'esempio successivo di calcoli di precision e recall mostra inoltre come, nel restituire i documenti, i valori di precision e recall non sono fissi! Dipende da che tipo di documento è stato restituito. Quindi man mano che si restituiscono documenti la precision e la recall cambiano in base all'ordine, per poi stabilizzarsi solo una volta restituiti tutti i documenti.

<img src="img/bibo.png" width="400px">

### Rank-Based Measures
Dobbiamo ora definire delle metriche che **prendano in considerazione anche l'ordine dei documenti restituiti**. Infatti nei motori di ricerca l'ordine dei risultati è fondamentale, in quanto gli utenti tendono a cliccare principalmente sui primi risultati restituiti -> è importante assicurarsi che i primi risultati restituiti siano quelli più rilevanti.

Esistono in generale due modi per rappresentare la rilevanza di un documento:
- **Relevance binaria**: per ogni documento restituito, si indica se è rilevante o non rilevante (1 o 0), senza sfumature
- **Relevance graduata**: per ogni documento restituito, si indica un punteggio di rilevanza (ad esempio da 0 a 3, dove 0 == non rilevante, 1 == poco rilevante, 2 == abbastanza rilevante, 3 == molto rilevante)

Nel caso di Relevance binaria si usano metriche come **Precision@k** (precision calcolata sui primi k documenti restituiti) e **Recall@k** (recall calcolata sui primi k documenti restituiti), o **MRR**. Nel caso di Relevance graduata si usano metriche come **NDCG@k** (Normalized Discounted Cumulative Gain), che tiene conto sia del grado di rilevanza che dell'ordine dei documenti restituiti. 

(Intuitivamente, con la binaria si dice se un documento serve o meno, mentre con la graduata si dice quanto un documento serve)

#### Precision@k e Recall@k
Per **Precision@k** si calcola la precision considerando solo i primi k documenti restituiti: $$Precision@k = \frac{TP@k}{TP@k + FP@k} = \frac{\text{numero di rilevanti nei primi K}}{K}$$ dove TP@k == true positives tra i primi k documenti restituiti, FP@k == false positives tra i primi k documenti restituiti.

Precision@k risponde quindi alla domanda: **Tra i primi k documenti restituiti, quanti sono effettivamente rilevanti?** Si ignorano quindi del tutto i documenti restituiti dopo il k-esimo. 

<img src="img/prec_k.png" width="400px">

Si può definire in modo analogo **Recall@k**: $$Recall@k = \frac{TP@k}{TP + FN} = \frac{\text{numero di rilevanti nei primi K}}{\text{numero totale di rilevanti}}$$

Ha senso usare queste metriche proprio per guardare solo ai primi K risultati restituiti dal sistema IR, in quanto è lì che gli utenti tendono a cliccare. Se ad esempio un sistema IR restituisce 100 documenti, ma gli utenti cliccano solo sui primi 10, allora è più importante avere alta precision e recall nei primi 10 documenti restituiti, piuttosto che nei successivi 90.

A partire da queste metriche è possibile costruire il **Recall-Precision Graph**:

<img src="img/grrr.png" width="500px">

Il grafico è calcolato semplicemente come segue: data una query, si scorre il ranking dei documenti restituito. Per ogni documento restituito, si calcolano precision e recall considerando solo i documenti restituiti fino a quel punto, per questo si usano Precision@k e Recall@k.

Tuttavia, **per ogni query si ottiene una curva diversa, bisogna mediare**. 

Prima di farlo però è conveniente **interpolare** la curva, in quanto tipicamente la curva di precision recall non è monotona. Questo avviene perché ad esempio quando si trova un documento irrilevante, la precision cala, ma se se ne trovano poi dei rilevanti essa risale. La recall invece è sempre crescente, in quanto il numero totale di documenti rilevanti (denominatore) è fisso (mentre per precision il denominatore dipende da FP che varia in base a quello che estraggo). 

Per fare interpolazione e trasformare la curva irregolare in una monotona, si applica la formula $$P(R) = \max\{\, P' : R' \ge R \wedge (R',P') \in S \,\}$$ dove S è l'insieme dei punti osservati (R, P). Si sta cioè dicendo che la precision interpolata a un certo livello di recall R è data dalla massima precision osservata in qualsiasi punto con recall maggiore o uguale a R.

In pratica con l'interpolazione, se scelgo un valore di recall ad esempio R = 0.4, guardo tutti i punti nella curva con almeno 0.4 di recall e prendo la precision massima tra quei punti. Facendo così ottengo una curva monotona decrescente a gradini.

<img src="img/interp.png" width="400px">

Una volta ottenuta la curva interpolata per ogni query, si può fare la media tra tutte le curve per ottenere una curva che rappresenta la performance del sistema IR su tutte le query. La curva ha sull'asse x la recall e sull'asse y la precision media -> **se medio su 50 query la curva finale mi dirà il comportamento medio del sistema IR su 50 query**

<img src ="img/interp2.png" width="400px">

A questo punto è possibile confrontare diversi sistemi IR guardando le loro curve interpolated precision-recall: **il sistema IR migliore è quello che ha la curva più alta** (quindi **con precision più alta a parità di recall**, nell'esempio Stem).

<img src="img/comp.png" width="400px">

Il **breakeven point** della curva è il punto per cui **Precision == Recall**. Questo punto è interessante in quanto rappresenta un compromesso di massimo equilibro tra le due. Oggi il breakeven point è usato meno, e come metriche oggi sono molto più usate MAP, MRR e NDCG.

<img src="img/bp.png" width="400px">

Qui di seguito una spiegazione del perché si deve fare l'interpolazione nel Recall-Precision Graph

<img src="img/interp2.png" width="400px">